# OpenTitans: Quick Start Guide

Welcome to OpenTitans! This notebook will guide you through the basics of using the OpenTitans framework for memory-augmented sequence modeling.

OpenTitans implements several variants of the Titans architecture:
- **MAC (Memory As Context)**: Uses a persistent memory as a contextual prefix.
- **MAG (Memory As Gate)**: Uses a gating mechanism to integrate long-term memory.
- **MAL (Memory As Layer)**: Integrates memory as a dedicated layer.
- **ATLAS**: Advanced Test-time Learning for Augmented Sequences.

## 1. Setup

First, let's make sure we have the necessary dependencies installed. (Assuming you have already installed `open-titans` or are running in the repo root).

In [ ]:
import torch
from open_titans import TitansMACModel, TitansMACConfig
from open_titans import TitansMAGModel, TitansMAGConfig

# Check if CUDA is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Using TitansMAC (Memory As Context)

TitansMAC segments the input and provides a "long-term memory" prefix that the model can attend to.

In [ ]:
# Initialize Configuration
mac_config = TitansMACConfig(
    vocab_size=1000,
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=8,
    segment_len=64,
    num_longterm_mem_tokens=16
)

# Initialize Model
mac_model = TitansMACModel(mac_config).to(device)

# Sample input (Batch size 2, Sequence length 128)
input_ids = torch.randint(0, 1000, (2, 128)).to(device)

# Forward pass
output = mac_model(input_ids)

print(f"Logits shape: {output.logits.shape}") # Should be (2, 128, 1000)

## 3. Using TitansMAG (Memory As Gate)

TitansMAG uses a dual-branch mechanism where one branch is a neural memory that gates the output of a sliding window attention.

In [ ]:
# Initialize Configuration
mag_config = TitansMAGConfig(
    vocab_size=1000,
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=8,
    window_size=32,
    neural_memory_segment_len=64
)

# Initialize Model
mag_model = TitansMAGModel(mag_config).to(device)

# Forward pass
output = mag_model(input_ids)

print(f"Logits shape: {output.logits.shape}")

## 4. Training and Loss

You can also compute the loss directly by passing `return_loss=True`.

In [ ]:
# Compute loss for the MAC model
output_with_loss = mac_model(input_ids, return_loss=True)
print(f"Loss: {output_with_loss.loss.item():.4f}")

## 5. Summary

This was a quick introduction to OpenTitans. 

Key points:
- All models follow a similar interface compatible with HuggingFace-style `PreTrainedModel`.
- Configurations allow tuning of memory-specific parameters like `segment_len` or `num_longterm_mem_tokens`.
- Models support both standard attention and memory-augmented pathways.